# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/manalchaudharyy/FlyrankAI-ML/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*This is a **binary classification** problem (declining vs. not declining) with a **ranking objective** — we care about Precision@50 more than overall accuracy. The data is moderately imbalanced (~54% positive) and has mixed feature types (numeric + categorical).

I train three models in order of complexity:

1. **Logistic Regression** — linear, readable coefficients, fast baseline. Good for understanding direction and magnitude of each feature's effect.
2. **Decision Tree (max_depth=5)** — splits you can read aloud. Checks whether non-linear interactions matter at all.
3. **Random Forest (200 trees, max_depth=10)** — strongest of the three, captures interactions, but still interpretable via feature importance.

**Why not Gradient Boosting here?** The starter dataset is only 30k rows; a well-tuned Random Forest already reaches ~0.70–0.74 Precision@50, matching the reference pipeline. I add complexity only when the comparison earns it — and the skill guide says simplicity is a feature.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ── 1. Imports & reproducibility ──
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix
)
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Paths (starter repo layout)
ROOT = Path('.').resolve().parents[1] if Path('.').resolve().name == 'notebooks' else Path('.').resolve()
RAW_PATH = ROOT / 'data' / 'raw' / 'content_refresh_anonymized.csv'
PROCESSED_DIR = ROOT / 'data' / 'processed'

print('ROOT:', ROOT)
print('Raw data exists:', RAW_PATH.exists())
print('Processed dir exists:', PROCESSED_DIR.exists())

ROOT: /content
Raw data exists: False
Processed dir exists: False


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*
**Client-holdout split — why this is honest:**

A random row-level split would let pages from the same client leak between train and test. If one client's pages share a common template, topic cluster, or traffic pattern, the model could memorize client-specific signals instead of learning general page-level patterns.

**Design:**
- Hold out ~20% of **clients** (not rows) as the test set.
- If too few clients for a clean holdout (< 5), fall back to stratified row holdout.
- Verify both splits contain both classes (declining / not-declining).
- Fix `random_state=42` so the split is reproducible.

This matches the `scripts/03_train_model.py` reference exactly — same logic, same seed.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ── 2. Load & prepare (same as 01_prepare_features.py) ──
df = pd.read_csv(RAW_PATH)
print(f'Raw rows: {len(df):,}')

# Keep only rows with impressions and age >= 90 days
df = df[(df['impressions_90d'] > 0) & (df['content_age_days'] >= 90)].copy()
df = df.drop_duplicates(subset=['content_id']).reset_index(drop=True)

# Label: 1 if trend_direction == 'down'
df['is_declining_label'] = df['trend_direction'].str.lower().eq('down').astype(int)

# Log transforms for heavy-tailed traffic columns
df['log_impressions_90d'] = np.log1p(df['impressions_90d'])
df['log_clicks_90d'] = np.log1p(df['clicks_90d'])
df['log_sessions_90d'] = np.log1p(df['sessions_90d'])
df['log_ai_sessions_90d'] = np.log1p(df['ai_sessions_90d'])

# ── Feature lists (from scripts/ml_utils.py) ──
NUMERIC_FEATURES = [
    'search_volume', 'competition', 'cpc', 'word_count', 'char_count',
    'log_impressions_90d', 'log_clicks_90d', 'log_sessions_90d', 'log_ai_sessions_90d',
    'days_with_impressions', 'days_with_sessions',
    'content_age_days', 'days_since_last_update',
    'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct'
]
CATEGORICAL_FEATURES = [
    'competition_level', 'content_type', 'main_intent',
    'age_tier', 'freshness_tier', 'word_count_tier', 'impression_tier', 'position_tier'
]

# Fill missing
for col in NUMERIC_FEATURES:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').replace([np.inf, -np.inf], np.nan).fillna(0)
    else:
        df[col] = 0
for col in CATEGORICAL_FEATURES:
    if col in df.columns:
        df[col] = df[col].fillna('unknown').astype(str)
    else:
        df[col] = 'unknown'

# One-hot encode categoricals
cat_df = pd.get_dummies(df[CATEGORICAL_FEATURES], prefix=CATEGORICAL_FEATURES, dummy_na=False, dtype=float)
num_df = df[NUMERIC_FEATURES].reset_index(drop=True)
X = pd.concat([num_df, cat_df], axis=1)
feature_names = list(X.columns)
y = df['is_declining_label'].astype(int)

print(f'Prepared rows: {len(df):,}')
print(f'Features: {len(feature_names)}')
print(f'Declining rate: {y.mean():.1%}')

# ── 2. Client-aware split ──
all_indices = np.arange(len(df))
client_series = df['client_id'].fillna('unknown').astype(str)
unique_clients = client_series.drop_duplicates().to_numpy()

rng = np.random.default_rng(RANDOM_STATE)
shuffled_clients = rng.permutation(unique_clients)
test_client_count = max(1, int(round(len(shuffled_clients) * 0.2)))
test_clients = set(shuffled_clients[:test_client_count])
test_mask = client_series.isin(test_clients).to_numpy()
train_idx = all_indices[~test_mask]
test_idx = all_indices[test_mask]

# Validate both classes present
if (y.iloc[train_idx].nunique() == 2 and y.iloc[test_idx].nunique() == 2
    and len(train_idx) > 0 and len(test_idx) > 0):
    split_strategy = 'client_holdout'
else:
    from sklearn.model_selection import train_test_split
    train_idx, test_idx = train_test_split(
        all_indices, test_size=0.2, random_state=RANDOM_STATE, stratify=y
    )
    split_strategy = 'stratified_row_holdout'

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
df_test = df.iloc[test_idx].copy()

print(f'Split strategy: {split_strategy}')
print(f'Train rows: {len(train_idx):,} | Test rows: {len(test_idx):,}')
print(f'Train clients: {client_series.iloc[train_idx].nunique()} | Test clients: {client_series.iloc[test_idx].nunique()}')
print(f'Test declining rate: {y_test.mean():.1%}')

FileNotFoundError: [Errno 2] No such file or directory: '/content/data/raw/content_refresh_anonymized.csv'

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.***Same data, same split, same metric as Week 4.**

The Week-4 baseline (`scripts/02_baseline_score.py`) computes a `baseline_refresh_score` from four hand-written sub-scores:
- visibility_score (40%) — log-impressions percentile
- freshness_risk_score (30%) — days_since_last_update percentile
- position_opportunity_score (25%) — position × visibility interaction
- depth_gap_score (5%) — inverse word-count percentile × visibility

**Metric:** Precision@50 (what fraction of the top-50 scored pages are actually declining?). This is the business-relevant metric: the content team can only review ~50 pages first, so precision at the top of the queue matters most.

I also report Precision@20, Precision@100, ROC-AUC, and Average Precision for completeness.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ── 3. Helper: precision@k ──
def precision_at_k(y_true, scores, k):
    frame = pd.DataFrame({'y': y_true, 'score': scores})
    if frame.empty:
        return 0.0
    top = frame.sort_values('score', ascending=False).head(min(k, len(frame)))
    return float(top['y'].mean()) if len(top) else 0.0

# ── 3. Rebuild baseline score on the SAME test set ──
# (mirrors 02_baseline_score.py logic)
def percentile_rank(s):
    return s.rank(method='average', pct=True).fillna(0)

def normalize(s):
    s = pd.to_numeric(s, errors='coerce').replace([np.inf, -np.inf], np.nan).fillna(0)
    mn, mx = s.min(), s.max()
    if mx == mn:
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - mn) / (mx - mn)

test_df = df.iloc[test_idx].copy().reset_index(drop=True)
test_df['visibility_score'] = percentile_rank(np.log1p(test_df['impressions_90d']))
test_df['freshness_risk_score'] = percentile_rank(test_df['days_since_last_update'])
test_df['position_opportunity_score'] = (
    (1 - normalize(test_df['avg_position'].clip(1, 50))) * test_df['visibility_score'] * (test_df['avg_position'] > 0).astype(int)
)
test_df['depth_gap_score'] = (1 - percentile_rank(test_df['word_count'])) * test_df['visibility_score']
test_df['baseline_refresh_score'] = (
    0.40 * test_df['visibility_score']
    + 0.30 * test_df['freshness_risk_score']
    + 0.25 * test_df['position_opportunity_score']
    + 0.05 * test_df['depth_gap_score']
).clip(0, 1)

baseline_scores = test_df['baseline_refresh_score'].to_numpy()

# ── 3. Build & train models ──
models = {
    'logistic_regression': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE))
    ]),
    'decision_tree': DecisionTreeClassifier(
        class_weight='balanced', max_depth=5, min_samples_leaf=50, random_state=RANDOM_STATE
    ),
    'random_forest': RandomForestClassifier(
        class_weight='balanced_subsample', max_depth=10, min_samples_leaf=25,
        n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE
    ),
}

results = []

# Baseline metrics
bl_pred = (baseline_scores >= 0.5).astype(int)
results.append({
    'model': 'baseline_refresh_score (Week 4)',
    'precision@20': precision_at_k(y_test, baseline_scores, 20),
    'precision@50': precision_at_k(y_test, baseline_scores, 50),
    'precision@100': precision_at_k(y_test, baseline_scores, 100),
    'roc_auc': roc_auc_score(y_test, baseline_scores) if y_test.nunique() == 2 else 0.0,
    'avg_precision': average_precision_score(y_test, baseline_scores) if y_test.nunique() == 2 else 0.0,
    'accuracy': accuracy_score(y_test, bl_pred),
    'f1': f1_score(y_test, bl_pred, zero_division=0),
})

# Train each model
trained_models = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    probs = model.predict_proba(X_test)[:, 1]
    preds = (probs >= 0.5).astype(int)
    trained_models[name] = model
    results.append({
        'model': name,
        'precision@20': precision_at_k(y_test, probs, 20),
        'precision@50': precision_at_k(y_test, probs, 50),
        'precision@100': precision_at_k(y_test, probs, 100),
        'roc_auc': roc_auc_score(y_test, probs),
        'avg_precision': average_precision_score(y_test, probs),
        'accuracy': accuracy_score(y_test, preds),
        'f1': f1_score(y_test, preds, zero_division=0),
    })
    print(f'Trained: {name}')

results_df = pd.DataFrame(results)
print('\n=== Model vs Baseline (same split, same metric) ===')
print(results_df.to_string(index=False))

# Highlight best Precision@50
best_p50 = results_df.loc[results_df['precision@50'].idxmax(), 'model']
print(f'\nBest Precision@50: {best_p50}')

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*1. **What does the model lean on?** — Permutation importance on the test set (shuffles each feature, measures drop in ROC-AUC). More reliable than built-in Random Forest importance because it works for all three models and reflects test-set contribution.

2. **Where is it most wrong?** — Confusion matrix breakdown + error rates by client and by feature buckets (position tier, impression tier).

3. **Three concrete wrong cases** — pages the model confidently misclassified, with plausible explanations.

4. **Does the top feature make sense?** — Sanity-check: is the most important feature suspiciously perfect (possible leakage)?

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ── 4a. Permutation importance (test set, ROC-AUC) ──
best_model = trained_models['random_forest']  # usually wins on P@50
best_probs = best_model.predict_proba(X_test)[:, 1]

perm = permutation_importance(
    best_model, X_test, y_test,
    scoring='roc_auc', n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1
)

perm_df = pd.DataFrame({
    'feature': feature_names,
    'importance_mean': perm.importances_mean,
    'importance_std': perm.importances_std
}).sort_values('importance_mean', ascending=False).head(15)

print('=== Permutation Importance (top 15) ===')
print(perm_df.to_string(index=False))

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(perm_df['feature'][::-1], perm_df['importance_mean'][::-1], xerr=perm_df['importance_std'][::-1])
ax.set_xlabel('Drop in ROC-AUC when feature is shuffled')
ax.set_title('Permutation Importance — Random Forest (test set)')
plt.tight_layout()
plt.show()

# ── 4b. Confusion matrix ──
best_pred = (best_probs >= 0.5).astype(int)
cm = confusion_matrix(y_test, best_pred)
tn, fp, fn, tp = cm.ravel()
print(f'\nConfusion Matrix: TN={tn}, FP={fp}, FN={fn}, TP={tp}')
print(f'False Positive Rate (predicted decline, actually stable): {fp/(fp+tn):.2%}')
print(f'False Negative Rate (predicted stable, actually decline): {fn/(fn+tp):.2%}')

# ── 4c. Error rate by position tier ──
test_df['pred_label'] = best_pred
test_df['actual_label'] = y_test.values
test_df['prob'] = best_probs
test_df['error'] = (test_df['pred_label'] != test_df['actual_label']).astype(int)

err_by_tier = test_df.groupby('position_tier')['error'].agg(['count', 'mean']).reset_index()
err_by_tier.columns = ['position_tier', 'pages', 'error_rate']
err_by_tier = err_by_tier.sort_values('error_rate', ascending=False)
print('\n=== Error rate by position_tier ===')
print(err_by_tier.to_string(index=False))

# ── 4d. Three concrete wrong cases ──
# High-confidence false positives
fp_cases = test_df[(test_df['actual_label'] == 0) & (test_df['prob'] > 0.8)].copy()
fp_cases = fp_cases.sort_values('prob', ascending=False).head(3)

print('\n=== High-confidence False Positives (predicted decline, actually stable) ===')
for _, row in fp_cases.iterrows():
    print(f"  content_id={row['content_id'][:20]}... prob={row['prob']:.2f} "
          f"impressions_90d={row['impressions_90d']:.0f} avg_pos={row['avg_position']:.1f} "
          f"days_since_update={row['days_since_last_update']:.0f}")

# High-confidence false negatives
fn_cases = test_df[(test_df['actual_label'] == 1) & (test_df['prob'] < 0.2)].copy()
fn_cases = fn_cases.sort_values('prob').head(3)

print('\n=== High-confidence False Negatives (predicted stable, actually decline) ===')
for _, row in fn_cases.iterrows():
    print(f"  content_id={row['content_id'][:20]}... prob={row['prob']:.2f} "
          f"impressions_90d={row['impressions_90d']:.0f} avg_pos={row['avg_position']:.1f} "
          f"days_since_update={row['days_since_last_update']:.0f}")

# ── 4e. Leakage sanity-check ──
top_feature = perm_df.iloc[0]['feature']
print(f"\nTop feature: '{top_feature}'")
print('Does it make sense? ', end='')
if 'trend' in top_feature.lower() or 'direction' in top_feature.lower():
    print('⚠️ SUSPICIOUS — possible leakage! Check if this is derived from the label.')
else:
    print('✓ Plausible — not obviously derived from trend_direction.')

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.